In [ ]:
!pip install trimesh rembg onnxruntime transformers torch torchvision pillow opencv-python



In [ ]:
!pip install trimesh rembg transformers torch torchvision pillow opencv-python


In [ ]:
from google.colab import files

uploaded = files.upload()
image_path = list(uploaded.keys())[0]

main(image_path)



Saving table.png to table.png


NameError: name 'main' is not defined

In [ ]:
import cv2
import torch
import numpy as np
import trimesh
from PIL import Image
from rembg import remove
from transformers import DPTForDepthEstimation, DPTImageProcessor
from google.colab import files

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def remove_background(image_path):
    img = Image.open(image_path).convert("RGBA")
    out = remove(img)
    out_path = "nobg.png"
    out.save(out_path)
    return out_path


def estimate_depth(image_path):
    processor = DPTImageProcessor.from_pretrained("Intel/dpt-hybrid-midas")
    model = DPTForDepthEstimation.from_pretrained(
        "Intel/dpt-hybrid-midas"
    ).to(DEVICE)
    model.eval()

    img = Image.open(image_path).convert("RGB")
    inputs = processor(images=img, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        depth = model(**inputs).predicted_depth

    depth = torch.nn.functional.interpolate(
        depth.unsqueeze(1),
        size=img.size[::-1],
        mode="bicubic",
        align_corners=False,
    ).squeeze()

    depth = depth.cpu().numpy()
    depth = (depth - depth.min()) / (depth.max() - depth.min())
    return depth, np.array(img)


def depth_to_mesh(depth, image, scale=120):
    h, w = depth.shape
    vertices = []
    faces = []
    colors = []

    for y in range(h):
        for x in range(w):
            z = depth[y, x] * scale
            vertices.append([x, y, z])
            colors.append(image[y, x] / 255.0)

    def vid(x, y):
        return y * w + x

    for y in range(h - 1):
        for x in range(w - 1):
            faces.append([vid(x, y), vid(x + 1, y), vid(x, y + 1)])
            faces.append([vid(x + 1, y), vid(x + 1, y + 1), vid(x, y + 1)])

    mesh = trimesh.Trimesh(
        vertices=np.array(vertices),
        faces=np.array(faces),
        vertex_colors=np.array(colors),
        process=False
    )
    return mesh


def main():
    print("📂 Upload a furniture image")
    uploaded = files.upload()
    image_path = list(uploaded.keys())[0]

    print("Removing background...")
    clean_img = remove_background(image_path)

    print("Estimating depth...")
    depth, image = estimate_depth(clean_img)

    print("Generating 3D mesh...")
    mesh = depth_to_mesh(depth, image)

    print("Saving OBJ file...")
    mesh.export("furniture_3d.obj")

    print("✅ Done! Output saved as furniture_3d.obj")


# ▶️ Run
main()


In [ ]:
!pip install torch torchvision torchaudio



In [ ]:
!pip install transformers timm pillow trimesh

In [ ]:
import torch
import numpy as np
import trimesh
from PIL import Image
from transformers import DPTForDepthEstimation, DPTImageProcessor
from google.colab import files

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def upload_image():
    print("📤 Upload a furniture image")
    uploaded = files.upload()
    image_path = list(uploaded.keys())[0]
    return image_path


def estimate_depth(image_path):
    processor = DPTImageProcessor.from_pretrained("Intel/dpt-hybrid-midas")
    model = DPTForDepthEstimation.from_pretrained(
        "Intel/dpt-hybrid-midas"
    ).to(DEVICE)
    model.eval()

    image = Image.open(image_path).convert("RGB")

    inputs = processor(images=image, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        depth = model(**inputs).predicted_depth

    depth = torch.nn.functional.interpolate(
        depth.unsqueeze(1),
        size=image.size[::-1],
        mode="bicubic",
        align_corners=False,
    ).squeeze()

    depth = depth.cpu().numpy()
    depth = (depth - depth.min()) / (depth.max() - depth.min())

    return depth, np.array(image)


def depth_to_mesh(depth, image, scale=150):
    h, w = depth.shape
    vertices = []
    faces = []
    colors = []

    for y in range(h):
        for x in range(w):
            z = depth[y, x] * scale
            vertices.append([x, y, z])
            colors.append(image[y, x] / 255.0)

    def vid(x, y):
        return y * w + x

    for y in range(h - 1):
        for x in range(w - 1):
            faces.append([vid(x, y), vid(x + 1, y), vid(x, y + 1)])
            faces.append([vid(x + 1, y), vid(x + 1, y + 1), vid(x, y + 1)])

    mesh = trimesh.Trimesh(
        vertices=np.array(vertices),
        faces=np.array(faces),
        vertex_colors=np.array(colors),
        process=False
    )

    return mesh


def main():
    image_path = upload_image()

    print("🧠 Estimating depth using MiDaS...")
    depth, image = estimate_depth(image_path)

    print("🧩 Creating 3D mesh...")
    mesh = depth_to_mesh(depth, image)

    print("💾 Saving OBJ file...")
    mesh.export("furniture_midas_3d.obj")

    print("✅ Done! File saved as furniture_midas_3d.obj")


if __name__ == "__main__":
    main()


📤 Upload a furniture image


Saving image 2.jpeg to image 2 (1).jpeg
🧠 Estimating depth using MiDaS...
🧩 Creating 3D mesh...
💾 Saving OBJ file...
✅ Done! File saved as furniture_midas_3d.obj


In [ ]:
import torch
import numpy as np
import trimesh
from PIL import Image
from transformers import DPTForDepthEstimation, DPTImageProcessor
from rembg import remove
from google.colab import files

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def upload_image():
    print("📤 Upload furniture image")
    uploaded = files.upload()
    image_path = list(uploaded.keys())[0]
    return image_path


def remove_background(image_path):
    image = Image.open(image_path).convert("RGBA")
    no_bg = remove(image)
    output_path = "nobg.png"
    no_bg.save(output_path)
    return output_path


def estimate_depth(image_path):
    processor = DPTImageProcessor.from_pretrained("Intel/dpt-hybrid-midas")
    model = DPTForDepthEstimation.from_pretrained(
        "Intel/dpt-hybrid-midas"
    ).to(DEVICE)
    model.eval()

    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        depth = model(**inputs).predicted_depth

    depth = torch.nn.functional.interpolate(
        depth.unsqueeze(1),
        size=image.size[::-1],
        mode="bicubic",
        align_corners=False,
    ).squeeze()

    depth = depth.cpu().numpy()
    depth = (depth - depth.min()) / (depth.max() - depth.min())

    return depth, np.array(image)


def depth_to_mesh(depth, image, scale=150):
    h, w = depth.shape
    vertices = []
    faces = []
    colors = []

    for y in range(h):
        for x in range(w):
            z = depth[y, x] * scale
            vertices.append([x, y, z])
            colors.append(image[y, x] / 255.0)

    def vid(x, y):
        return y * w + x

    for y in range(h - 1):
        for x in range(w - 1):
            faces.append([vid(x, y), vid(x + 1, y), vid(x, y + 1)])
            faces.append([vid(x + 1, y), vid(x + 1, y + 1), vid(x, y + 1)])

    mesh = trimesh.Trimesh(
        vertices=np.array(vertices),
        faces=np.array(faces),
        vertex_colors=np.array(colors),
        process=False
    )
    return mesh


def main():
    image_path = upload_image()

    print("🧹 Removing background...")
    clean_image = remove_background(image_path)

    print("🧠 Estimating depth...")
    depth, image = estimate_depth(clean_image)

    print("🧩 Generating 3D mesh...")
    mesh = depth_to_mesh(depth, image)

    print("💾 Saving OBJ...")
    mesh.export("furniture_2_5D.obj")

    print("✅ Done! Output saved as furniture_2_5D.obj")


if __name__ == "__main__":
    main()
